# Rendering — đọc Windows từ OperatingSystem
Chạy ô code bên dưới, chọn **Benchmark final.xlsx**, rồi chờ tải ZIP kết quả.
Giữ nguyên thuật toán, 100 mẫu ACCEPT, chia 80 train/20 test và KFold của notebook gốc. Bản này chỉ chuẩn hóa cách đọc Windows; chưa thay đổi cách đánh giá theo nhóm CPU–GPU.


In [ ]:
# RENDER: đọc OperatingSystem, không đọc Windows từ ReviewNote.
# Dán toàn bộ file này vào MỘT ô code Google Colab rồi chạy.
# Khi hiện nút chọn tệp, tải lên Benchmark final.xlsx.
# Giữ nguyên mô hình, chia 80/20 và KFold của notebook gốc để đối chiếu.
# Vẫn yêu cầu đúng 100 mẫu ACCEPT như bản gốc.
# Cách chia theo nhóm CPU-GPU chưa được áp dụng trong bản sửa này.

import io
import re
import math
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import files
print('Chọn Benchmark final.xlsx (có cột OperatingSystem).')
uploaded = files.upload()
excel_names = [name for name in uploaded if name.lower().endswith('.xlsx')]
if len(uploaded) != 1 or len(excel_names) != 1:
    raise ValueError('Hãy tải lên đúng một file Excel .xlsx.')
filename = excel_names[0]
excel = pd.ExcelFile(io.BytesIO(uploaded[filename]))
print('File đang dùng:', filename)
render = pd.read_excel(excel, sheet_name='Benchmark_Rendering')
reference = pd.read_excel(excel, sheet_name='ReferenceBenchmark')
render.columns = render.columns.str.strip()
reference.columns = reference.columns.str.strip()
print('\n===== DỮ LIỆU GỐC =====')
print('Số dòng Benchmark_Rendering:', len(render))
print('Số dòng ReferenceBenchmark:', len(reference))

def parse_windows(value):
    if pd.isna(value):
        raise ValueError('Thiếu OperatingSystem. Hãy nhập Win 10 hoặc Win 11.')
    normalized = re.sub('\\s+', ' ', str(value).strip().lower())
    mapping = {'win 10': 0, 'windows 10': 0, 'win 11': 1, 'windows 11': 1}
    if normalized not in mapping:
        raise ValueError(f'OperatingSystem không hợp lệ: {value!r}. Chỉ nhận Win 10/Win 11 hoặc Windows 10/Windows 11.')
    return mapping[normalized]
if 'OperatingSystem' not in render.columns:
    raise ValueError('Sheet Benchmark_Rendering thiếu cột OperatingSystem. Hãy dùng Benchmark final.xlsx.')

def normalize_cpu(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = name.split('@')[0]
    name = name.replace('processor', '')
    name = re.sub('[^a-z0-9]', '', name)
    return name

def normalize_gpu(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = name.replace('nvidia', '')
    name = re.sub('[^a-z0-9]', '', name)
    return name

def get_gpu_generation(name):
    if pd.isna(name):
        return np.nan
    text = str(name).lower()
    match = re.search('rtx\\s*(\\d{2})\\d{2}', text)
    if match is None:
        return np.nan
    return float(match.group(1))
ref_cpu = reference.loc[reference['ComponentType'].astype(str).str.strip().eq('CPU') & reference['ReviewStatus'].astype(str).str.strip().eq('ACCEPT') & reference['Metric'].astype(str).str.strip().eq('CPU Mark')].copy()
ref_cpu['CpuKey'] = ref_cpu['ModelName'].map(normalize_cpu)
ref_cpu['RawScore'] = pd.to_numeric(ref_cpu['RawScore'], errors='coerce')
assert not ref_cpu['CpuKey'].duplicated().any(), 'ReferenceBenchmark có CPU bị trùng'
cpu_lookup = ref_cpu.set_index('CpuKey')['RawScore'].rename('CpuMultiScore')
ref_gpu = reference.loc[reference['ComponentType'].astype(str).str.strip().eq('GPU') & reference['ReviewStatus'].astype(str).str.strip().eq('ACCEPT') & reference['Metric'].astype(str).str.strip().eq('G3D Mark')].copy()
ref_gpu['GpuKey'] = ref_gpu['ModelName'].map(normalize_gpu)
ref_gpu['RawScore'] = pd.to_numeric(ref_gpu['RawScore'], errors='coerce')
assert not ref_gpu['GpuKey'].duplicated().any(), 'ReferenceBenchmark có GPU bị trùng'
gpu_lookup = ref_gpu.set_index('GpuKey')['RawScore'].rename('GpuScore')
data = render.loc[render['ReviewStatus'].astype(str).str.strip().eq('ACCEPT') & render['TestName'].astype(str).str.strip().eq('PCMark 10') & render['Metric'].astype(str).str.strip().eq('RENDER')].copy()
data = data.reset_index(drop=True)
print('\nSố mẫu Rendering hợp lệ:', len(data))
if len(data) != 100:
    raise ValueError(f'Dataset không có đúng 100 mẫu. Hiện tại có {len(data)} mẫu.')
data['CpuKey'] = data['CpuModel'].map(normalize_cpu)
data['GpuKey'] = data['GpuModel'].map(normalize_gpu)
data['CpuMultiScore'] = data['CpuKey'].map(cpu_lookup)
data['GpuScore'] = data['GpuKey'].map(gpu_lookup)
data['IsIntel'] = data['CpuModel'].astype(str).str.contains('intel', case=False, na=False).astype(int)
data['IsUltra'] = data['CpuModel'].astype(str).str.contains('ultra', case=False, na=False).astype(int)
data['GpuGeneration'] = data['GpuModel'].map(get_gpu_generation)
data['IsWindows11'] = data['OperatingSystem'].map(parse_windows)
data['Score'] = pd.to_numeric(data['Score'], errors='coerce')
FEATURES = ['GpuScore', 'CpuMultiScore', 'IsIntel', 'GpuGeneration', 'IsUltra', 'IsWindows11']
required_cols = FEATURES + ['Score']
print('\n===== MISSING VALUE =====')
print(data[required_cols].isna().sum())
if data[required_cols].isna().any().any():
    error_rows = data.loc[data[required_cols].isna().any(axis=1), ['RunId', 'CpuModel', 'GpuModel'] + required_cols]
    print(error_rows)
    raise ValueError('Có CPU/GPU chưa nối được ReferenceBenchmark.')
assert not data['RunId'].duplicated().any(), 'Có RunId bị trùng'
X = data[FEATURES].copy()
y = data['Score'].copy()
print('\n===== INPUT X =====')
print(FEATURES)
print('\n===== TARGET =====')
print('Score = PCMark 10 RENDER')
all_index = np.arange(len(data))
train_idx, test_idx = train_test_split(all_index, test_size=20, random_state=42, shuffle=True)
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
train_data = data.iloc[train_idx].copy()
test_data = data.iloc[test_idx].copy()
print('\n===== CHIA DỮ LIỆU =====')
print('Train:', len(X_train))
print('Test:', len(X_test))
assert len(X_train) == 80
assert len(X_test) == 20
print('\n===== 20 MẪU TEST =====')
print(test_data[['RunId', 'CpuModel', 'GpuModel', 'Score']].sort_values('RunId').to_string(index=False))
model = LinearRegression()
model.fit(X_train, y_train)
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}
train_result = evaluate(y_train, train_pred)
test_result = evaluate(y_test, test_pred)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
oof_pred = cross_val_predict(LinearRegression(), X_train, y_train, cv=cv, method='predict')
oof_result = evaluate(y_train, oof_pred)
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_result = evaluate(y_test, baseline_pred)
print('\n=============================================')
print('              KẾT QUẢ MODEL')
print('=============================================')
print(f"TRAIN 80 mẫu\nMAE  = {train_result['MAE']:.3f}\nRMSE = {train_result['RMSE']:.3f}\nR2   = {train_result['R2']:.6f}")
print('\n---------------------------------------------')
print(f"5-FOLD OOF trên 80 TRAIN\nMAE  = {oof_result['MAE']:.3f}\nRMSE = {oof_result['RMSE']:.3f}\nR2   = {oof_result['R2']:.6f}")
print('\n---------------------------------------------')
print(f"TEST 20 mẫu\nMAE  = {test_result['MAE']:.3f}\nRMSE = {test_result['RMSE']:.3f}\nR2   = {test_result['R2']:.6f}")
print('\n---------------------------------------------')
print(f"BASELINE TEST\nMAE  = {baseline_result['MAE']:.3f}\nRMSE = {baseline_result['RMSE']:.3f}\nR2   = {baseline_result['R2']:.6f}")
print('\n===== INTERCEPT =====')
print(model.intercept_)
print('\n===== COEFFICIENT =====')
for feature, coef in zip(FEATURES, model.coef_):
    print(feature, '=', coef)
print('\n===== CÔNG THỨC =====')
formula = f'R_hat = {model.intercept_:.12f}'
for feature, coef in zip(FEATURES, model.coef_):
    if coef >= 0:
        formula += f'\n + {coef:.12f} * {feature}'
    else:
        formula += f'\n - {abs(coef):.12f} * {feature}'
print(formula)
comparison = test_data[['RunId', 'CpuModel', 'GpuModel', 'Score']].copy()
comparison['PredictedScore'] = test_pred
comparison['Error'] = comparison['Score'] - comparison['PredictedScore']
comparison['AbsoluteError'] = comparison['Error'].abs()
comparison = comparison.sort_values('AbsoluteError', ascending=False)
print('\n===== CHI TIẾT TEST =====')
print(comparison.round(2).to_string(index=False))

def predict_rendering(cpu_model, gpu_model, operating_system='Windows 11'):
    cpu_key = normalize_cpu(cpu_model)
    gpu_key = normalize_gpu(gpu_model)
    if cpu_key not in cpu_lookup.index:
        raise ValueError('CPU chưa tồn tại trong ReferenceBenchmark: ' + str(cpu_model))
    if gpu_key not in gpu_lookup.index:
        raise ValueError('GPU chưa tồn tại trong ReferenceBenchmark: ' + str(gpu_model))
    cpu_score = float(cpu_lookup.loc[cpu_key])
    gpu_score = float(gpu_lookup.loc[gpu_key])
    gpu_gen = get_gpu_generation(gpu_model)
    is_intel = int('intel' in str(cpu_model).lower())
    is_ultra = int('ultra' in str(cpu_model).lower())
    is_windows11 = parse_windows(operating_system)
    new_data = pd.DataFrame([{'GpuScore': gpu_score, 'CpuMultiScore': cpu_score, 'IsIntel': is_intel, 'GpuGeneration': gpu_gen, 'IsUltra': is_ultra, 'IsWindows11': is_windows11}])
    new_data = new_data[FEATURES]
    prediction = model.predict(new_data)[0]
    return float(prediction)
example = predict_rendering(cpu_model='Intel Core i7-14700K', gpu_model='NVIDIA GeForce RTX 4070 SUPER', operating_system='Windows 11')
print('\n===== VÍ DỤ =====')
print('i7-14700K + RTX 4070 SUPER + Windows 11')
print('Rendering Score dự đoán:', round(example, 2))

# XUẤT GÓI MÔ HÌNH
import json
import zipfile
import joblib
import sklearn
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
created_at = datetime.now(timezone.utc)
version = created_at.strftime('%Y%m%dT%H%M%S%fZ')
output_dir = Path(f'rendering_model_{version}')
output_dir.mkdir(exist_ok=False)
excel_name = Path(filename).name
if 'uploaded' in globals() and filename in uploaded:
    excel_bytes = uploaded[filename]
else:
    excel_bytes = Path(filename).read_bytes()
excel_sha256 = hashlib.sha256(excel_bytes).hexdigest()
intercept = float(model.intercept_)
coefficients = {feature: float(coef) for feature, coef in zip(FEATURES, model.coef_)}
formula = 'RenderingScore = intercept'
for feature in FEATURES:
    formula += f" + coefficients['{feature}'] * {feature}"
training_ranges = {feature: {'min': float(train_data[feature].min()), 'max': float(train_data[feature].max())} for feature in FEATURES}
model_info = {'model_version': f'rendering-linear-v1-{version}', 'created_at_utc': created_at.isoformat(), 'model_type': 'Multiple Linear Regression', 'algorithm': 'sklearn.linear_model.LinearRegression', 'target': 'PCMark 10 Rendering and Visualization Score', 'metric': 'RENDER', 'fit_scope': '80 training rows only', 'training_row_count': int(len(train_data)), 'test_row_count': int(len(test_data)), 'split': {'method': 'train_test_split', 'train_size': 80, 'test_size': 20, 'shuffle': True, 'random_state': 42}, 'feature_order': FEATURES, 'feature_definitions': {'GpuScore': 'PassMark G3D Mark from ReferenceBenchmark', 'CpuMultiScore': 'PassMark CPU Mark from ReferenceBenchmark', 'IsIntel': 'Intel CPU = 1; other CPU = 0', 'GpuGeneration': 'RTX 30xx=30, RTX 40xx=40, RTX 50xx=50', 'IsUltra': 'Intel Core Ultra = 1; otherwise 0', 'IsWindows11': 'Windows 11 = 1; Windows 10 = 0'}, 'formula': formula, 'intercept': intercept, 'coefficients': coefficients, 'normalization_rules': {'cpu': {'lowercase': True, 'remove_text_after_at_sign': True, 'remove_word': 'processor', 'keep_only': 'a-z0-9'}, 'gpu': {'lowercase': True, 'remove_word': 'nvidia', 'keep_only': 'a-z0-9'}}, 'feature_engineering': {'IsIntel': "1 if CpuModel contains 'intel' ignoring case", 'IsUltra': "1 if CpuModel contains 'ultra' ignoring case", 'GpuGeneration': 'regex rtx\\s*(\\d{2})\\d{2}', 'IsWindows11': 'Windows 11 = 1; Windows 10 = 0'}, 'input_requirements': {'missing_values': 'Reject; do not replace with zero', 'cpu_reference_required': True, 'gpu_reference_required': True, 'outside_training_ranges': 'Flag as extrapolation'}, 'training_ranges': training_ranges, 'metrics': {'training': {'mae': float(train_result['MAE']), 'rmse': float(train_result['RMSE']), 'r2': float(train_result['R2'])}, 'cross_validation_oof': {'mae': float(oof_result['MAE']), 'rmse': float(oof_result['RMSE']), 'r2': float(oof_result['R2'])}, 'test': {'mae': float(test_result['MAE']), 'rmse': float(test_result['RMSE']), 'r2': float(test_result['R2'])}}, 'training_cpu_count': int(train_data['CpuKey'].nunique()), 'training_gpu_count': int(train_data['GpuKey'].nunique()), 'source_excel_name': excel_name, 'source_excel_sha256': excel_sha256, 'versions': {'sklearn': sklearn.__version__, 'numpy': np.__version__, 'pandas': pd.__version__, 'joblib': joblib.__version__}, 'limitations': ['Training R2 is not independent validation.', 'The model was fitted on 80 rows.', 'The remaining 20 rows were reserved for testing.', 'Benchmark run conditions can vary.', 'IsIntel, IsUltra and IsWindows11 coefficients represent statistical associations, not causality.', 'Prediction target is PCMark 10 Rendering and Visualization Score.', 'Predictions outside training ranges are extrapolation.']}
model_info['feature_engineering']['IsWindows11'] = 'Read OperatingSystem; Win 11/Windows 11 = 1; Win 10/Windows 10 = 0; reject missing or unknown values'
model_info['input_requirements']['operating_system_required'] = True
joblib.dump(model, output_dir / 'model.joblib')
info_path = output_dir / 'model_info.json'
info_path.write_text(json.dumps(model_info, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
train_data.to_csv(output_dir / 'data_used.csv', index=False, encoding='utf-8-sig')
reference.to_csv(output_dir / 'reference.csv', index=False, encoding='utf-8-sig')
test_predictions = test_data[['RunId', 'CpuModel', 'GpuModel', 'Score'] + FEATURES].copy()
test_predictions['PredictedScore'] = test_pred
test_predictions['Error'] = test_predictions['Score'] - test_predictions['PredictedScore']
test_predictions['AbsoluteError'] = test_predictions['Error'].abs()
test_predictions.to_csv(output_dir / 'test_predictions.csv', index=False, encoding='utf-8-sig')
loaded_model = joblib.load(output_dir / 'model.joblib')
loaded_info = json.loads(info_path.read_text(encoding='utf-8'))
loaded_test = pd.read_csv(output_dir / 'test_predictions.csv')
expected = loaded_test['PredictedScore'].to_numpy(dtype=float)
joblib_predictions = loaded_model.predict(loaded_test[loaded_info['feature_order']])
np.testing.assert_allclose(joblib_predictions, expected, rtol=1e-10, atol=1e-08)
print('Đạt: model.joblib đọc lại chính xác.')
formula_predictions = np.full(len(loaded_test), float(loaded_info['intercept']), dtype=float)
for feature in loaded_info['feature_order']:
    coef = float(loaded_info['coefficients'][feature])
    formula_predictions += coef * loaded_test[feature].to_numpy(dtype=float)
np.testing.assert_allclose(formula_predictions, expected, rtol=1e-10, atol=1e-08)
print('Đạt: công thức JSON khớp model Python.')
export_names = ['data_used.csv', 'model.joblib', 'model_info.json', 'reference.csv', 'test_predictions.csv']
zip_path = Path(f'rendering_model_{version}.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in export_names:
        archive.write(output_dir / name, arcname=name)
print('\n======================================')
print('XUẤT MODEL RENDERING THÀNH CÔNG')
print('======================================')
print('Model version:', model_info['model_version'])
print('Train:', len(train_data), 'mẫu')
print('Test:', len(test_data), 'mẫu')
print('Train R2:', model_info['metrics']['training']['r2'])
print('OOF R2:', model_info['metrics']['cross_validation_oof']['r2'])
print('Test R2:', model_info['metrics']['test']['r2'])
print('\nCác file:')
for name in export_names:
    print('-', name)
print('\nGói ZIP:', zip_path.name)
from google.colab import files
files.download(str(zip_path))
